In [11]:
from dotenv import load_dotenv
from tqdm import tqdm
import textgrad as tg
from textgrad.tasks import DataLoader
import numpy as np
load_dotenv(override=True)
import concurrent.futures
from common import (compute_spec_score, 
                    clean_output, 
                    set_seed, 
                    load_data, 
                    BASELINE_PROMPT,
                    EVAL_FN_DESCRIPTION, 
                    BATCH_SIZE, 
                    MAX_STEPS, 
                    NUM_WORKERS)


In [12]:

def eval_dataset(test_set, model):

    def process_example(ex):
        input = tg.Variable(ex, requires_grad=False, role_description="query to the language model")
        response = model(input)
        return compute_spec_score(clean_output(str(response)))

    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        results = list(tqdm(executor.map(process_example, test_set), total=len(test_set)))

    return results  # accuracy_list is now `results`


In [13]:
def run_validation_revert(system_prompt: tg.Variable, results, model, val_set):

    val_performance = np.mean(eval_dataset(val_set, model)) # Compute val performance
    previous_performance = np.mean(results["validation_acc"][-1]) # Retrieve last val performance from results
    print("val_performance: ", val_performance)
    print("previous_performance: ", previous_performance)
    previous_prompt = results["prompt"][-1]
    
    if val_performance < previous_performance:
        #print(f"rejected prompt: {system_prompt.value}")
        system_prompt.set_value(previous_prompt)
        val_performance = previous_performance

    results["validation_acc"].append(val_performance)

In [14]:
#set_seed(42)

llm_api_eval = tg.get_engine(engine_name="gpt-4o")
llm_api_test = tg.get_engine(engine_name="gpt-4o")
tg.set_backward_engine(llm_api_eval, override=True)

# Load the data and the evaluation function
train_set, val_set, test_set = load_data()
print("Train/Val/Test Set Lengths: ", len(train_set), len(val_set), len(test_set))

Train/Val/Test Set Lengths:  20 20 20


In [ ]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)

system_prompt = tg.Variable(BASELINE_PROMPT, 
                            requires_grad=True, 
                            role_description="system prompt to the language model")

model_evaluation = tg.BlackboxLLM(llm_api_eval, system_prompt)

system_prompt = tg.Variable(BASELINE_PROMPT, 
                            requires_grad=True,
                            role_description="structured system prompt to a somewhat capable language model that specifies the behavior and strategies for the QA task")
model = tg.BlackboxLLM(llm_api_test, system_prompt)

optimizer = tg.TextualGradientDescent(engine=llm_api_eval, parameters=[system_prompt])

results = {"test_acc": [], "prompt": [], "validation_acc": []}
results["test_acc"].append(eval_dataset(test_set, model))
results["validation_acc"].append(eval_dataset(val_set, model))
results["prompt"].append(system_prompt.get_value())

print(f"Validation score: {np.mean(results['validation_acc'][-1])}")
print(f"Test score: {np.mean(results['test_acc'][-1])}")

100%|██████████| 20/20 [00:00<00:00, 71.15it/s]


In [17]:
# The system prompt that will guide the behavior of the loss function.
loss_system_prompt = "You are a smart language model that evaluates NuSMV controllers to specific tasks. You do not solve problems or propose new code snippets, only evaluate existing solutions critically and give very concise feedback."
loss_system_prompt = tg.Variable(loss_system_prompt, requires_grad=False, role_description="system prompt to the loss function")

with open("examples/sample_ltl.txt", 'r', encoding='utf-8') as file:
    specs = file.read()

# The instruction that will be the prefix
instruction = f"Think about the task and the NuSMV snippet. Score description: {EVAL_FN_DESCRIPTION}"

# The format string and setting up the call
format_string = "{instruction}\nTask: {{problem}}\nSolution: {{code}}\nScore: {{score}}"
format_string = format_string.format(instruction=instruction)

fields = {"problem": None, "code": None, "score": None}
formatted_llm_call = tg.autograd.FormattedLLMCall(engine=llm_api_eval,
                                                  format_string=format_string,
                                                  fields=fields,
                                                  system_prompt=loss_system_prompt)

# Finally, the loss function
def loss_fn(problem: tg.Variable, code: tg.Variable, score: float) -> tg.Variable:
    inputs = {"problem": problem, "code": code, "score": score}
    
    return formatted_llm_call(inputs=inputs,
                              response_role_description=f"evaluation of the NuSMV. Score description: {EVAL_FN_DESCRIPTION}")

In [ ]:
step_count = 0
stop = False
while not stop:
    for batch_x in train_loader:
        print(f"Training step {step_count + 1}.")
        optimizer.zero_grad()
        losses = []
        
        for x in tqdm(batch_x):
            x = tg.Variable(str(x), requires_grad=False, role_description="query to the language model")
            response = model(x)
            score = compute_spec_score(clean_output(str(response)))
            score = tg.Variable(str(score), requires_grad=False, role_description="specifications score")
            eval_output_variable = loss_fn(x, response, score)
            print(f"training example score {score}", flush=True)
            losses.append(eval_output_variable)

        total_loss = tg.sum(losses)
        total_loss.backward()
        optimizer.step()
        
        run_validation_revert(system_prompt, results, model, val_set)
        
        print("sys prompt: ", system_prompt)
        test_acc = eval_dataset(test_set, model)
        results["test_acc"].append(test_acc)
        results["prompt"].append(system_prompt.get_value())
        step_count += 1
        
        if step_count == MAX_STEPS:
            stop = True
            break


In [ ]:
np.mean(results["validation_acc"][-1]), np.mean(results["test_acc"][-1])